# Import Required Libraries
Import the necessary libraries, including the dataset loader from src/dataloaders/datasets/a_thaliana_dataset.py.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np

In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import TrainingArguments, Trainer, logging
import torch
from datasets import Dataset

# list of HyenaDNA pretrained models available in the hugging face hub
pretrained_models_max_length = {
    'hyenadna-tiny-1k-seqlen-hf': 1024,
        'hyenadna-small-32k-seqlen-hf': 32768,
        'hyenadna-medium-160k-seqlen-hf': 160000,
        'hyenadna-medium-450k-seqlen-hf': 450000,  # T4 up to here
        'hyenadna-large-1m-seqlen-hf': 1_000_000,
}
pretrained_model_name = 'hyenadna-tiny-1k-seqlen-hf'

# instantiate pretrained model
checkpoint = 'LongSafari/'+pretrained_model_name
tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True)

/home/lpedraza/anaconda3/envs/hyena-dna/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/lpedraza/anaconda3/envs/hyena-dna/lib/python3.8/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Explicitly passing a `revision` is encouraged when loading a model with custom code to ensure no malicious code has been contributed in a newer revision.


# Load Dataset
Load the dataset using the provided BED and FA files.

In [3]:
# Import the dataset loader
from src.dataloaders.datasets.a_thaliana_dataset import a_thalinana_Dataset

# Define the paths to the BED and FA files
bed_file_path = 'A_Thaliana.bed'
fa_file_path = 'data/hg38/hg38.ml.fa'

# Load the dataset using the provided BED and FA files
ds_test = a_thalinana_Dataset(bed_file=bed_file_path, 
                              fasta_file=fa_file_path, 
                              split='test', 
                              max_length = 500,
                              tokenizer = tokenizer,
                              tokenizer_name='char'
                              )

In [4]:
ds_test.label_dict

{'gene': 0, 'intergenic': 1}

In [5]:
data_test = []
labels_test = []
for _, (d, l) in enumerate(ds_test):
    data_test.append(d.tolist())
    labels_test.append(l.item())

In [6]:
len(data_test), len(labels_test)

(374, 374)

In [7]:
seq_0 = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(data_test[0]))
print(seq_0)
print(len(seq_0))

TGAATGAATTTTGAGTTAGGGAATATAAATGATAACTGGATGGCATTTTACAGAATCCTTGGGTGTCCCAGTTATTTAGAACAGTGAGTCCTACACTGAAAACAATAGGAAAACACTCTTATAAGCCATACCTTCATTTTGCCAATTAAATTTTATTATTAAATAATTTTTTCCTATGTTTTCTAAAAGAGATAAGACTGAATGAAACAATCATCCTAAAGAGAAAAGCTAGAATTGTGTAGTGGCATCAGGCTTACTAGTAACTCTAAATACTACTATGTCGTGGCGATTAACCTGTTTATAGAGGATCCTATCTTTTGTTCAAGACTTATCAGGAGTTGGTTCTAACATTCAGTTAGTTTGCCCTGAAATATTGAATTCATGCTCAGAGCAGTGAGAAAAGAGTGTATTCAAATAAGTTGAAAGAGAAAACATTTAGTGTTTTGTTTTGTTTGTTTGTTTGTTTTGCTTTTTTAGAAAACATTAAAGAATCAGGAATC
500


# Display Dataset Information
Display basic information about the dataset, such as the number of entries and a preview of the data.

In [11]:
# Display basic information about the dataset
# Number of entries in the dataset
num_entries = len(ds_test)
print(f"Number of entries in the dataset: {num_entries}")

Number of entries in the dataset: 353


In [42]:
seq_0 = tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(data_test[1]))
print(seq_0)
print(len(seq_0))

CTTTGAGCTCTTTATATAATTAAAAATTAACCCCCTCAGCCAGGTGTGGCAGCTCACACCTGTAATCCCAGCATTTTGGAAGGCTGAGGTGAGAGAACTGCCTGAGTGTAGGAGATCACCACCAACCTGGTCAACATAGTGACACTTTGTCTCTACTAAAAATTAAAAAAAAAAATGAGCTACACGTTGCAGTGCACACCTGTAGTCCGAGCTACTGGGGAGGCTAAGACTGGAGGATCACTTGAGTCTAGAAGGTTGAGGCTGCAGTAAGCTATGATCACACCATTGCACTTTAGCTTTGCTAAGAGCAAGACTGCATTTCTTAAACAAAATAAAAATTAGATGGGAATATTGCTCAAGCCCTGGAGGTTGAGGCTGCAGTTAACTGTGATTGCACCACTGCAGTCCAGCCTAGGTGATAGAGCAAGACCCTTTCTCTAAAAATAAAATAAAATAAAAATTAACCTTCTATCATATTTCCCAGTAACACCTTCCCTCCT
500


In [43]:
from Bio import SeqIO
from Bio.Seq import Seq

def find_sequence_in_fasta(fasta_file, query_sequence):
    seq_len = len(query_sequence)
    # Cargar el archivo FASTA
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequence = str(record.seq)
        reverse_complement_seq = str(record.seq.reverse_complement())
        chr_len = len(sequence)

        # Buscar la secuencia directa
        if query_sequence in sequence:
            start = sequence.find(query_sequence)
            end = start + seq_len
            print(f"Secuencia encontrada en {record.id} (strand +) en la posición: {start}")
            print(f"{record.id}, seq in strand +: {sequence[start: end]}")
            return (record.id, start, end, '+')

        # Buscar el reverse complement
        if query_sequence in reverse_complement_seq:
            end = chr_len - reverse_complement_seq.find(query_sequence)
            start = end - seq_len
            print(f"Secuencia encontrada en {record.id} (strand -) en la posición: {start} del strand +")
            print(f"{record.id}, seq in strand +: {reverse_complement_seq[start: end]}")
            return (record.id, start, end, '-')

    print("Secuencia no encontrada en el archivo FASTA.")
    return None

# Buscar la secuencia
result = find_sequence_in_fasta(fasta_file=fa_file_path, query_sequence=seq_0)
if result:
    chr, start, end, strand = result
    print(f"Coordenadas: chr={chr}, start={start}, end={end}, strand={strand}")

Secuencia encontrada en chr1 (strand -) en la posición: 756262 del strand +
chr1, seq in strand +: AGTAGGTCTGCAATATTACTCCAAGAAGGAAGCTGTCTTGTCAAGTTAGCTTGTTCCTATGCTTCTTTTTAAGTTGGCAAAGTGTTCTTTACATTTGAGAATCATTCCTTCCCAGTCACCAAATTGACATAAAATGTTGAATAATTCTACTCCATCATTTATCCATTTCCACCACATATGTTTTAAACTCCACTGAATCATGAAGCATTAAAACAAAATTGAATAACAAGTGTAAATCTACATATCCCTCTGAATTTTCTCAGTTACATTAGTTTAGGCTGAACTTATATAAATGCACTATTATTTGTGTTGTATTTTTCACCTCCTTCATATCCATGTGCTAAAATTGTCTTCATTGAGTTTGCCCTGTCACGGAGAATATGGCAAGGTATATCATAAATTTATAAAAATTTAGAAGTCACAAAATTGAAAATATTGACTATTACATTGCCCTGTACCTTTAGTATTAAGATCCATGTGAATATTTATTCAATAAAATT
Coordenadas: chr=chr1, start=756262, end=756762, strand=-


In [44]:
# carga el archivo A_Thaliana.bed en un dataframe
df = pd.read_csv(bed_file_path, sep='\t', header=None)
df.columns = ['chr', 'start', 'end', 'label', 'split', 'strand']
df.head()

,chr,start,end,label,split,strand
0,chr1,50000,50100,gene,2,-
1,chr2,519826,525528,intergenic,0,.
2,chr1,756262,757300,intergenic,2,-
3,chr4,759357,768230,intergenic,2,.
4,chr5,723043,723288,gene,1,+


In [ ]:
from Bio import SeqIO
from BCBio import GFF

def gff_to_bed(gff_file, bed_file):
    with open(gff_file) as gff_handle, open(bed_file, 'w') as bed_handle:
        for rec in GFF.parse(gff_handle):
            for feature in rec.features:
                if feature.type == "gene":
                    chrom = rec.id
                    start = feature.location.start
                    end = feature.location.end
                    name = feature.qualifiers.get("Name", [""])[0]
                    score = 0
                    strand = feature.strand
                    bed_handle.write(f"{chrom}\t{start}\t{end}\t{name}\t{score}\t{strand}\n")

# Uso de la función
gff_file = "path/to/your_file.gff"
bed_file = "path/to/your_file.bed"
gff_to_bed(gff_file, bed_file)